In [0]:
from pyspark.sql import functions as F

# 1. Hent data fra Bronze
df = spark.table(
    "dataengineering_bronze.raw_data.butiksordrer"
)

# 2. Fjern dubletter
df = df.dropDuplicates()

# 3. Fjern rækker, hvor alle værdier er NULL
df = df.na.drop(how="all")

# 4. Fjern mellemrum fra tekstfelter
for col_name, dtype in df.dtypes:
    if dtype == "string":
        df = df.withColumn(
            col_name,
            F.trim(F.col(col_name))
        )

# 5. Opret Silver-katalog og schema
spark.sql("""
    CREATE CATALOG IF NOT EXISTS dataengineering_silver
""")

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS
    dataengineering_silver.cleaned_data
""")

# 6. Gem som Delta Table
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "dataengineering_silver.cleaned_data.butiksordrer"
    )

display(df)